## **1st Trial: Enhanced 14%**

In [66]:
import pandas as pd
import zipfile

In [67]:
ZIP_PATH = './Dataset/customers-2000000.zip'
CSV_FILENAME = 'customers-2000000.csv'
parse_dates = ['Subscription Date']

Before diving into memory optimization, we first inspect a small sample of the dataset using peek = pd.read_csv(nrows=5). This step helps us understand the structure of the data, including column names, data types, and initial formatting. From this preview, we can identify that the dataset contains multiple categorical and textual fields such as Customer Id, Company, Country, Email, and Website (the raw dataset consumes significantly higher memory per column), along with an integer-based Index column.

In [68]:
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    with z.open(CSV_FILENAME) as f:
        peek = pd.read_csv(f, nrows=5)
        print(peek.columns.tolist())
        print('\n')
        print(peek.dtypes)
        print('\n')
        print(peek.head())

['Index', 'Customer Id', 'First Name', 'Last Name', 'Company', 'City', 'Country', 'Phone 1', 'Phone 2', 'Email', 'Subscription Date', 'Website']


Index                 int64
Customer Id          object
First Name           object
Last Name            object
Company              object
City                 object
Country              object
Phone 1              object
Phone 2              object
Email                object
Subscription Date    object
Website              object
dtype: object


   Index      Customer Id First Name Last Name                  Company  \
0      1  4962fdbE6Bfee6D        Pam    Sparks             Patel-Deleon   
1      2  9b12Ae76fdBc9bE       Gina     Rocha  Acosta, Paul and Barber   
2      3  39edFd2F60C85BC    Kristie     Greer                Ochoa PLC   
3      4  Fa42AE6a9aD39cE     Arthur    Fields               Moyer-Wang   
4      5  F5702Edae925F1D   Michelle   Blevins            Shah and Sons   

               City                               

Meanwhile, most of the columns are currently inferred as generic object types, which is a strong indicator that the dataset is not yet optimized for memory efficiency. This becomes an important consideration, especially because we are working with a large-scale dataset (2 million rows), where inefficient data types can significantly increase memory usage.

The function below compares memory usage between a raw and an optimized version of a CSV file loaded from a ZIP archive. It reads the same data twice: once with default data types and once with specified optimized data types and optional date parsing. Then it calculates memory usage for each column and prints a table showing the difference and percentage of memory saved. Columns that give a significant optimization up to 20% memory are flagged. Finally, it also shows total memory savings.

In [69]:
def compare_memory(zip_path, csv_filename, dtype_map, parse_dates=None, nrows=1000):
    with zipfile.ZipFile(zip_path, 'r') as z:
        with z.open(csv_filename) as f:
            raw = pd.read_csv(f, nrows=nrows)

        with z.open(csv_filename) as f:
            optimized = pd.read_csv(f, nrows=nrows, dtype=dtype_map, parse_dates=parse_dates or [])

    raw_detail = raw.memory_usage(deep=True).to_dict()
    optimized_detail = optimized.memory_usage(deep=True).to_dict()

    print(f"{'COLUMN':<25} {'RAW':>12} {'OPTIMIZED':>12} {'SAVED':>8}")
    print("-" * 65)
    for col in raw.columns:
        r = raw_detail[col]
        o = optimized_detail[col]
        saved = (1 - o/r) * 100
        flag = " [YES]" if saved > 20 else ""
        print(f"  {col:<23} {r:>12,} {o:>12,} {saved:>7.1f}%{flag}")

    print("-" * 65)
    raw_total = sum(raw_detail.values())
    optimized_total = sum(optimized_detail.values())
    savings = (1 - optimized_total / raw_total) * 100
    print(f"  {'TOTAL':<23} {raw_total:>12,} {optimized_total:>12,} {savings:>7.1f}%")

In [70]:
dtype_map = {
    'Index':'int32',
    'Customer Id':'string',
    'First Name':'category',
    'Last Name':'category',
    'Company':'string',
    'City':'string',
    'Country':'category',
    'Phone 1':'string',
    'Phone 2':'string',
    'Email':'string',
    'Website':'string',
}

The above data types were changed to reduce memory usage and make the dataset more efficient to process. For example:
- Index (int64 to int32) = because it does not need large integer storage.
- First Name, Last Name, and Country (object to string/category) = the category type is especially useful for columns with repeated values, because it stores each unique value only once, which saves a lot of memory. 

Overall, these changes make the dataset easier to handle when working with large amounts of data.

In [71]:
compare_memory(ZIP_PATH, CSV_FILENAME, dtype_map, parse_dates)

COLUMN                             RAW    OPTIMIZED    SAVED
-----------------------------------------------------------------
  Index                          8,000        4,000    50.0% [YES]
  Customer Id                   64,000       64,000     0.0%
  First Name                    54,790       47,169    13.9%
  Last Name                     55,041       52,753     4.2%
  Company                       65,316       65,316     0.0%
  City                          60,812       60,812     0.0%
  Country                       59,894       24,683    58.8% [YES]
  Phone 1                       65,309       65,309     0.0%
  Phone 2                       65,022       65,022     0.0%
  Email                         72,262       72,262     0.0%
  Subscription Date             59,000        8,000    86.4% [YES]
  Website                       71,964       71,964     0.0%
-----------------------------------------------------------------
  TOTAL                        701,410      601,290    14

Overall, we have reduced total usage from 701,410 bytes to 601,290 bytes, which is a 14.3% saving.

The biggest improvements come from specific columns like Subscription Date, Country, and Index. For example:
- Subscription Date (saves 86.4%) = because it was likely converted from a heavy object type into a more efficient datetime format.
- Country (saves 58.8%) = because it was converted to a category type, which is very efficient for repeated values.
- Index (saves 50%) = because it was downcast from int64 to int32.

However, some columns like Customer Id, Company, City, Phone, and Email show 0% savings because their data types were not significantly changed or they still require similar memory due to their unique text values.

## **2nd Trial: Enhanced 65.5%**

In [73]:
dtype_map_arrow = {
    'Index':            'int32',
    'Customer Id':      'string[pyarrow]',  
    'First Name':       'string[pyarrow]',   
    'Last Name':        'string[pyarrow]',
    'Company':          'string[pyarrow]',
    'City':             'string[pyarrow]',
    'Country':          'string[pyarrow]',
    'Phone 1':          'string[pyarrow]',
    'Phone 2':          'string[pyarrow]',
    'Email':            'string[pyarrow]',
    'Website':          'string[pyarrow]',
}

This dtype_map_arrow changes the data types to use the PyArrow backend for string columns. By switching to string[pyarrow], pandas uses Apache Arrow’s memory format, which is more optimized for handling large-scale string data. It is faster, uses less memory, and provides better support for modern data processing operations.

The numeric column Index is still kept as int32 because it is already an efficient integer type.

In [74]:
compare_memory(ZIP_PATH, CSV_FILENAME, dtype_map_arrow, parse_dates)

COLUMN                             RAW    OPTIMIZED    SAVED
-----------------------------------------------------------------
  Index                          8,000        4,000    50.0% [YES]
  Customer Id                   64,000       23,000    64.1% [YES]
  First Name                    54,790       13,790    74.8% [YES]
  Last Name                     55,041       14,041    74.5% [YES]
  Company                       65,316       24,316    62.8% [YES]
  City                          60,812       19,812    67.4% [YES]
  Country                       59,894       18,894    68.5% [YES]
  Phone 1                       65,309       24,309    62.8% [YES]
  Phone 2                       65,022       24,022    63.1% [YES]
  Email                         72,262       31,262    56.7% [YES]
  Subscription Date             59,000        8,000    86.4% [YES]
  Website                       71,964       30,964    57.0% [YES]
-----------------------------------------------------------------
  T

The result shows that using PyArrow data types greatly improves memory efficiency. Total memory usage drops from 701,410 bytes to 236,410 bytes, saving about 66.3%. Most string columns like names, city, and country save over 60–70% memory because string[pyarrow] is more compact than the default object type in pandas. Overall, this optimization makes the dataset much lighter and more efficient for large-scale processing.

## **3. Explain how it's different from splitting the small vs large files.**

The key difference between small and large CSV files lies in how they are loaded into memory and how they are processed. Small datasets such as 100K rows can be loaded entirely into memory at once using pd.read_csv(), which makes all pandas operations directly available, including filtering, grouping, and transformation. Since the memory usage is relatively low (around 50–200MB), the data can be explored freely without special optimization techniques. 

In contrast, large datasets such as 2 million rows cannot be safely loaded all at once because they may require several gigabytes of memory and risk causing out-of-memory errors. Because of this, large files are processed using chunking or streaming, where data is read and handled in smaller parts, and results are combined afterward.

In short, small files use eager loading where all data is immediately available, while large files rely on streaming or lazy processing to prevent memory overload.

#### **How it works (examples)**

##### **Small file (eager loading):**

In [77]:
df = pd.read_csv("./Dataset/customers-100000.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   Index              100000 non-null  int64 
 1   Customer Id        100000 non-null  object
 2   First Name         100000 non-null  object
 3   Last Name          100000 non-null  object
 4   Company            100000 non-null  object
 5   City               100000 non-null  object
 6   Country            100000 non-null  object
 7   Phone 1            100000 non-null  object
 8   Phone 2            100000 non-null  object
 9   Email              100000 non-null  object
 10  Subscription Date  100000 non-null  object
 11  Website            100000 non-null  object
dtypes: int64(1), object(11)
memory usage: 9.2+ MB


In [78]:
df["Country"].value_counts()
df.groupby("Company").size()
df.dropna()

,Index,Customer Id,First Name,Last Name,Company,City,Country,Phone 1,Phone 2,Email,Subscription Date,Website
0,1,ffeCAb7AbcB0f07,Jared,Jarvis,Sanchez-Fletcher,Hatfieldshire,Eritrea,274.188.8773x41185,001-215-760-4642x969,gabriellehartman@benjamin.com,2021-11-11,https://www.mccarthy.info/
1,2,b687FfC4F1600eC,Marie,Malone,Mckay PLC,Robertsonburgh,Botswana,283-236-9529,(189)129-8356x63741,kstafford@sexton.com,2021-05-14,http://www.reynolds.com/
2,3,9FF9ACbc69dcF9c,Elijah,Barrera,Marks and Sons,Kimbury,Barbados,8252703789,459-916-7241x0909,jeanettecross@brown.com,2021-03-17,https://neal.com/
3,4,b49edDB1295FF6E,Sheryl,Montgomery,"Kirby, Vaughn and Sanders",Briannaview,Antarctica (the territory South of 60 deg S),425.475.3586,(392)819-9063,thomassierra@barrett.com,2020-09-23,https://www.powell-bryan.com/
4,5,3dcCbFEB17CCf2E,Jeremy,Houston,Lester-Manning,South Brianna,Micronesia,+1-223-666-5313x4530,252-488-3850x692,rubenwatkins@jacobs-wallace.info,2020-09-18,https://www.carrillo.com/
...,...,...,...,...,...,...,...,...,...,...,...,...
99995,99996,67F24BEBAa16d1c,Dana,Winters,"Pham, Conner and Wade",Kirkfurt,Sierra Leone,820-930-7616,+1-061-779-5511x3267,ppittman@watson.com,2020-05-30,https://www.maynard.com/
99996,99997,17b1dbDaB2ad0fB,Gabriela,Pacheco,Fletcher-Hodge,Virginiahaven,Comoros,+1-480-464-8646,(015)822-1443x33211,murillojeremiah@mcmahon-medina.com,2022-04-09,http://www.eaton.com/
99997,99998,c586CFBA6fb9dcC,Mikayla,Hubbard,Austin Ltd,Sheriville,Mayotte,+1-567-149-3941x67118,361-900-3348,orhodes@harding-galloway.biz,2022-03-05,https://hayes.com/
99998,99999,bb6cb6AC9d0CAf7,Javier,Berg,Welch Inc,Stephenschester,Belarus,(381)105-4698,360-905-2308x301,rodneytanner@jennings-boone.net,2021-12-29,http://zuniga.com/


Here, all data is already in memory, so any operation can be applied directly.

##### **Large file (chunking / streaming):**

In [80]:
result = {}

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    with z.open(CSV_FILENAME) as f:
        
        for chunk in pd.read_csv(f, chunksize=100000):
            counts = chunk["Country"].value_counts()

            for country, count in counts.items():
                result[country] = result.get(country, 0) + count

print(result)

{'Korea': 16240, 'Congo': 16208, 'Switzerland': 8261, 'Kuwait': 8318, 'Netherlands': 8354, 'Saint Helena': 8076, 'Slovakia (Slovak Republic)': 8359, 'Cape Verde': 8293, 'United Kingdom': 8144, 'Saint Kitts and Nevis': 8349, 'United States Minor Outlying Islands': 8177, 'Iceland': 7965, 'French Polynesia': 8202, 'Ghana': 8228, 'Canada': 8169, 'Gibraltar': 8093, 'Bulgaria': 8154, 'Mozambique': 8006, 'Denmark': 8148, 'Portugal': 8293, 'Grenada': 8206, 'Central African Republic': 8180, 'Marshall Islands': 8128, 'Kiribati': 8210, 'Tuvalu': 8256, 'Oman': 8155, 'American Samoa': 8113, 'Malta': 8129, 'Vietnam': 8388, 'Comoros': 8172, 'Jordan': 8428, 'Monaco': 8213, 'Chile': 8268, 'Ecuador': 8177, 'San Marino': 8096, 'Burkina Faso': 8269, 'Greenland': 8106, 'Macao': 8247, 'Suriname': 8375, 'Saint Vincent and the Grenadines': 8239, 'Nigeria': 8246, 'Greece': 8198, 'Uzbekistan': 8263, 'Norfolk Island': 8113, 'China': 8202, 'Turkey': 8172, 'France': 8180, 'Nauru': 8092, 'Haiti': 8120, 'Afghanistan

Here, data is processed in parts. Each chunk is handled separately, and results are combined manually.

The key difference while splitting between small and large CSV processing is also clearly visible in execution time. The small dataset (100K rows) takes only about 0.5 seconds, because all data can be loaded directly into memory and processed immediately without any overhead.

In contrast, the large dataset (2 million rows) takes around 49.4 seconds, because the data must be read in chunks, processed step by step, and then combined, which adds extra I/O and computation overhead.

This shows that while small files prioritize speed and simplicity through full in-memory processing, large files prioritize stability and memory safety through chunked processing, even if it takes significantly more time.